# 05 · Experiment C (localisation) and Experiment D (quantification)

### What can and cannot be measured with RSNA-STR
RSNA-STR provides **slice-level** PE flags but **no voxel masks** and **no clot-volume measurements**. Therefore:

| Metric | Level at which it is computed here | Needs voxel masks? |
|---|---|---|
| IoU / Dice | along **z** (set of PE slices) — plus voxel-level functions ready for use if you obtain masks | voxel version only |
| Localisation sensitivity | slice-level and lesion-level (contiguous PE z-runs) | no |
| False-positive rate | predicted lesions per PE-negative study | no |
| Burden MAE / RMSE / r / calibration | against the **slice-extent proxy** (fraction of vessel-bearing slices with PE) | no — but it is a proxy, not true clot volume |

Do not describe these as voxel-level segmentation accuracy or as validated clot-burden agreement in the paper; state the proxy explicitly.

In [ ]:
%run ./00_common.ipynb

In [ ]:
sp = load_splits(); tr, va, te = sp["train"], sp["val"], sp["test"]
TH = json.load(open(f"{cfg.ckpt_dir}/thresholds.json")); cfg.thr_det, cfg.thr_slice = TH["thr_det"], TH["thr_slice"]
model = PENet(cfg, Flags()).to(DEVICE); model.load_state_dict(torch.load(f"{cfg.ckpt_dir}/penet_full.pt", map_location=DEVICE)); model.eval()

import pandas as pd, matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
R = predict(model, make_loader(te, cfg, False)); print(len(R["sid"]), "test studies | detection:", det_metrics(R))

## Experiment C — localisation

In [ ]:
def iou_dice(pred, gt, eps=1e-6):
    pred, gt = pred.astype(bool), gt.astype(bool); i = (pred & gt).sum()
    return float(i / (pred | gt).sum().clip(min=eps)) if (pred | gt).sum() else 1.0, float(2 * i / (pred.sum() + gt.sum()).clip(min=eps)) if (pred.sum() + gt.sum()) else 1.0
# (voxel-level iou_dice(pred_mask, gt_mask) works as-is if you have 3D reference masks, e.g. from a PE segmentation dataset.)

def lesion_stats(prob, lab, thr):
    P, G = runs(prob >= thr), runs(lab > .5)
    ov = lambda a, b: not (a[1] < b[0] or b[1] < a[0])
    tp = sum(any(ov(g, p) for p in P) for g in G); fp = sum(not any(ov(p, g) for g in G) for p in P)
    return tp, len(G), fp, len(P)

thr = cfg.thr_slice; m = (R["slice_valid"] > 0) & (R["y"][:, 0] > 0.5)           # PE+ studies with slice labels
ious, dices, sens = [], [], []
for p, l in zip(R["slice_prob"][m], R["slice_lab"][m]):
    i, d = iou_dice(p >= thr, l > .5); ious.append(i); dices.append(d); sens.append(((p >= thr) & (l > .5)).sum() / max((l > .5).sum(), 1))
tp = G = fpl = P = 0
for p, l in zip(R["slice_prob"][m], R["slice_lab"][m]): a, b_, c, d = lesion_stats(p, l, thr); tp += a; G += b_; fpl += c; P += d
C = dict(slice_IoU=np.mean(ious), slice_Dice=np.mean(dices), slice_sensitivity=np.mean(sens), lesion_sensitivity=tp / max(G, 1),
         lesion_precision=1 - fpl / max(P, 1), FP_lesions_per_PE_negative_study=fp_lesions_per_negative(R["slice_prob"], R["y"][:, 0], thr))
C = {k: round(float(v), 4) for k, v in C.items()}; print(C); json.dump(C, open(f"{cfg.out_dir}/expC_localisation.json", "w"), indent=1)

In [ ]:
# FROC-style curve: lesion sensitivity vs. false-positive lesions per PE-negative study
ths = np.linspace(0.05, 0.95, 19); S, Fp = [], []
for t in ths:
    tp = G = 0
    for p, l in zip(R["slice_prob"][m], R["slice_lab"][m]): a, b_, _, _ = lesion_stats(p, l, t); tp += a; G += b_
    S.append(tp / max(G, 1)); Fp.append(fp_lesions_per_negative(R["slice_prob"], R["y"][:, 0], t))
plt.figure(figsize=(4.5, 3.8)); plt.plot(Fp, S, "o-"); plt.xlabel("FP lesions / PE-negative study"); plt.ylabel("lesion sensitivity"); plt.title("FROC-style curve"); plt.grid(alpha=.3); plt.show()

## Experiment D — PE-burden quantification
Compared quantities: `burden_ext` (slice-extent index from the localisation branch), `burden_head` (direct regression head) and `burden_vol` (volumetric index = heat-map volume ÷ vessel volume; only weakly supervised through the MIL slice loss, so expect a different scale). Reference = slice-extent proxy `burden_t`. Evaluate on PE-positive studies.

In [ ]:
pos = R["y"][:, 0] > 0.5; ref = R["burden_t"][pos]; rows = []
for name in ("burden_ext", "burden_head", "burden_vol"):
    pr = R[name][pos]; r = pearsonr(pr, ref)[0] if pr.std() > 0 else np.nan
    rows.append(dict(estimator=name, MAE=np.abs(pr - ref).mean(), RMSE=np.sqrt(((pr - ref) ** 2).mean()), pearson=r, spearman=spearmanr(pr, ref)[0],
                     calib_slope=np.polyfit(pr, ref, 1)[0] if pr.std() > 0 else np.nan, calib_intercept=np.polyfit(pr, ref, 1)[1] if pr.std() > 0 else np.nan))
Dm = pd.DataFrame(rows).round(4); Dm.to_csv(f"{cfg.out_dir}/expD_quantification.csv", index=False); Dm

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4)); pr = R["burden_head"][pos]
ax[0].scatter(ref, pr, s=12, alpha=.6); ax[0].plot([0, ref.max()], [0, ref.max()], "k--"); ax[0].set_xlabel("proxy burden"); ax[0].set_ylabel("predicted"); ax[0].set_title("agreement")
mean, diff = (pr + ref) / 2, pr - ref; ax[1].scatter(mean, diff, s=12, alpha=.6)
for v in (diff.mean(), diff.mean() + 1.96 * diff.std(), diff.mean() - 1.96 * diff.std()): ax[1].axhline(v, color="r" if v != diff.mean() else "k", ls="--")
ax[1].set_title(f"Bland-Altman (bias {diff.mean():.3f}, LoA ±{1.96 * diff.std():.3f})"); ax[1].set_xlabel("mean"); ax[1].set_ylabel("pred - ref")
q = np.quantile(pr, np.linspace(0, 1, 6)); bins = np.digitize(pr, q[1:-1]); ax[2].plot([pr[bins == k].mean() for k in range(5)], [ref[bins == k].mean() for k in range(5)], "o-")
ax[2].plot([0, ref.max()], [0, ref.max()], "k--"); ax[2].set_title("calibration (quintiles)"); ax[2].set_xlabel("mean predicted"); ax[2].set_ylabel("mean observed"); plt.tight_layout(); plt.show()

from sklearn.metrics import cohen_kappa_score, confusion_matrix
cls = lambda x: np.digitize(x, cfg.burden_bins)                         # 0 low, 1 moderate, 2 high
print("Burden class kappa (quadratic):", round(cohen_kappa_score(cls(ref), cls(pr), weights="quadratic"), 3)); print(confusion_matrix(cls(ref), cls(pr), labels=[0, 1, 2]))
print("Reminder: 'low/moderate/high' here is a research-derived index, not a clinically validated severity score.")

### Severity heads (RV/LV ≥ 1, laterality, central, chronic) — AUC on test

In [ ]:
names = LABELS[1:]; out = {}
for j, n in enumerate(names):
    mk = R["y_mask"][:, 1 + j] > 0
    out[n] = safe_auc(R["y"][mk, 1 + j], R["sev_prob"][mk, j]) if mk.sum() else np.nan
pd.Series(out).round(4).rename("test AUC")